<div align="right" style="text-align: right"><i>Peter Norvig<br>Sept 2026</i></div>

# Integer Palindromes

Consider this slightly-modified version of a problem from the [2026 AIME](https://artofproblemsolving.com/wiki/index.php/2026_AIME_I_Problems/Problem_2?srsltid=AfmBOor4WAOSBdOAsLKKfku51DG21_h8NUzWEenLIieAMaUFWilw8QeB) (American Invitational Mathematics Examination): 

***Write a function to return the positive integer palindromes written in base 10, with no zero digits, and whose digits add up to N.***


Here are my thoughts about the problem:
- An integer can only become a [palindromic number](https://en.wikipedia.org/wiki/Palindromic_number) when expressed as a sequence of characters.
  - Thus, my function's signature will be `palindromes_with_digit_sum(N: int) -> list[str]`
  - For example, `palindromes_with_digit_sum(4) == ['1111',  '121', '22',  '4']`.
  - The empty string is [considered](https://en.wikipedia.org/wiki/Empty_string) a palindrome, so `palindromes_with_digit_sum(0) == ['']`.
- Valid palindromes come from two cases:
  - Base case: a palindrome can be empty (when *N* = 0) or can be a single digit (when 1 ≤ *N* ≤ 9).
  - Multi-digit case: a palindrome can have first and last digit *d*, and a middle part that is a palindrome.
    - Middle parts can be found by recursively asking for palindromes with digit sum equal to *N* - 2*d*.
    - Since the recursive step always subtracts 2*d* from *N*, the process will always terminate in at most *N*/2 steps.
    - (If zero digits were allowed, it would not terminate.)

# My Solution

Those thoughts easily translate into a function that is concise and reasonably efficient:

In [1]:
from functools import cache

@cache
def palindromes_with_digit_sum(N: int) -> list[str]:
    """Palindromic digit-strings, with no zero digits, whose digits sum to `N`."""
    base_case  = [''] if N == 0 else [str(N)] if 1 <= N <= 9 else []
    multi_case = [f'{d}{middle}{d}'
                  for d in range(1, 10) if d * 2 <= N
                  for middle in palindromes_with_digit_sum(N - 2 * d)]
    return multi_case + base_case

Here's the program in action:

In [2]:
palindromes_with_digit_sum(7)

['1111111', '11311', '12121', '151', '21112', '232', '313', '7']

 That looks right.
 
# Analysis

Here are some facts I noticed:
   - If *N* is odd, each palindrome must have an odd length, with an odd middle digit.
   - If *N* is even, each palindrome can have an even length, or an odd length with an even middle digit.
   - The number of solutions for *N* = 2*m* is the same as for *N* = 2*m* + 1. Here's why:
      - For each even-length 2*m* solution, you can insert 1 as the middle digit.
      - For each odd-length 2*m* solution, you can increase the (even) middle digit by 1.
   - The number of solutions for *N* + 2 is either double or slightly less than double the number of solutions for *N*.
      - The doubling breaks down when we run out of digits. For example, with *N* = 7, one solution is "7", and for *N* + 2 we can change that solution to either "171" or "9", doubling the count (and similarly for every other palindrome with sum 7). But for *N* = 8, we  we can't add 2 to the digit "8", so we get one less than double the number of solutions.

Exploring the counts of number of valid palindromes:

In [3]:
def count(N) -> int: return len(palindromes_with_digit_sum(N))

print(*(count(N) for N in range(30)))

1 1 2 2 4 4 8 8 16 16 31 31 62 62 124 124 248 248 496 496 991 991 1980 1980 3956 3956 7904 7904 15792 15792


In [4]:
# Does the number of solutions double from N to N+2?
{N: count(N + 2) / count(N) for N in range(30)}

{0: 2.0,
 1: 2.0,
 2: 2.0,
 3: 2.0,
 4: 2.0,
 5: 2.0,
 6: 2.0,
 7: 2.0,
 8: 1.9375,
 9: 1.9375,
 10: 2.0,
 11: 2.0,
 12: 2.0,
 13: 2.0,
 14: 2.0,
 15: 2.0,
 16: 2.0,
 17: 2.0,
 18: 1.997983870967742,
 19: 1.997983870967742,
 20: 1.997981836528759,
 21: 1.997981836528759,
 22: 1.997979797979798,
 23: 1.997979797979798,
 24: 1.9979777553083924,
 25: 1.9979777553083924,
 26: 1.9979757085020242,
 27: 1.9979757085020242,
 28: 1.9980369807497467,
 29: 1.9980369807497467}

# Tests

I'll define `test` to test any function that purports to meet the specification. The output of the function is canonicalized because an implementation might reasonably choose to use ints instead of strings, and might return (or yield) the palindromes in any order (not necessarily the lexicographic order that my program produces). If the function has a cache, it is cleared at the start.

In [5]:
import time

def test(palindrome_function) -> bool:
    """Tests for the palindrome digit sum problem."""

    if hasattr(palindrome_function, 'cache_clear'):
        palindrome_function.cache_clear()
    
    def f(N: int) -> list[str]: 
        """Canonicalize the output of palindrome_function(N)."""
        return sorted(map(str, palindrome_function(N)))

    # Tests for small values of N
    assert f(0) == [''] or f(0) == [] # Spec is unclear; either answer is acceptable
    assert f(1) == ['1']
    assert f(2) == ['11',                                                          '2']
    assert f(3) == ['111',                                                         '3']
    assert f(4) == ['1111',                      '121',     '22',                  '4']
    assert f(5) == ['11111',                     '131',    '212',                  '5']
    assert f(6) == ['111111',  '11211', '1221',  '141',   '2112',   '222',   '33', '6']
    assert f(7) == ['1111111', '11311', '12121', '151',  '21112',   '232',  '313', '7']

    # Try the function on some larger N and check the count of palindromes.
    for N, count in {38: 502_864, 40: 1_004_737, 42: 2_007_494}.items():
        assert len(f(N)) == count == len(f(N + 1))

It takes about a second to run the test suite on cases up to *N* = 44 (for which there are 4,011,032 valid palindromes).

# LLM Solutions

I asked four Large Language Model coding agents to solve this problem: Gemini 3.1 Pro, GPT 5.6 Sol, Claude Sonnet 5, and Kimi K3.

On the positive side, they all wrote correct code that passed the tests, and they all got the idea that you only have to build up the first half of the digit string,  and that you shouldn't generate half-strings with a sum greater than *N*/2.

On the negative side, they all ran 10 to 20 times slower on the test suite compared to my solution, largely because none of them used `@cache`. Only Gemini produced what I considered a clear, simple solution (basically the same as mine, but using `for` and `if` statements instead of conditional expressions and list comprehensions). All the others had overly complicated code (with a line count 6 times mine). Only GPT provided type hints.

For each coding agent, the table below lists the test pass/fail grade, the lines of code (including blank lines and comments), the wall time in seconds for the test suite, and my subjective code-quality judgment. So for this round, the human came out on top!

|Author|Test|LOC|Time|Subjective|
|-----|---:|---:|---:|---|
|Human|pass|10|0.7|Good|
|Gemini|pass|19|7.4|Good|
|ChatGPT|pass|60|10.1|Too complicated|
|Claude|pass|62|16.1|Too complicated|
|Kimi|pass|62|14.9|Too complicated|



# Gemini 3.1 Pro's Solution


In [6]:
def gemini_generate_palindromes(target_sum):
    """
    Yields all positive integer palindromes (as strings) with no zero digits
    whose digits add up to the target_sum.
    """
    # Base case 1: The remaining sum is exactly 0 (even-length palindrome center)
    if target_sum == 0:
        yield ""
        
    # Base case 2: The remaining sum is a single non-zero digit (odd-length center)
    if 1 <= target_sum <= 9:
        yield str(target_sum)
        
    # Recursive step: Wrap matching pairs of digits (1 through 9) around the outside
    for d in range(1, 10):
        # We need enough remaining sum for two 'd' digits
        if target_sum >= 2 * d:
            for inner in gemini_generate_palindromes(target_sum - 2 * d):
                yield f"{d}{inner}{d}"

# GPT 5.6 Sol's Solution

In [7]:
from collections.abc import Iterator


def _halves_with_sum(weights: tuple[int, ...], target: int) -> Iterator[str]:
    """Yield digit strings whose weighted digit sum is ``target``."""

    suffix_weight = [0] * (len(weights) + 1)
    for index in range(len(weights) - 1, -1, -1):
        suffix_weight[index] = suffix_weight[index + 1] + weights[index]

    digits: list[str] = []

    def visit(index: int, remaining: int) -> Iterator[str]:
        if index == len(weights):
            if remaining == 0:
                yield "".join(digits)
            return

        weight = weights[index]
        rest_weight = suffix_weight[index + 1]
        for digit in range(1, 10):
            after = remaining - weight * digit
            # Every remaining digit is in [1, 9].
            if rest_weight <= after <= 9 * rest_weight:
                digits.append(str(digit))
                yield from visit(index + 1, after)
                digits.pop()

    yield from visit(0, target)


def chatGPT_palindromes_with_digit_sum(n: int) -> Iterator[int]:
    """Yield positive zero-free decimal palindromes with digit sum ``n``.

    Results are yielded in increasing numeric order.  The sequence is finite:
    since every digit is at least one, a result can contain at most ``n``
    digits.
    """

    if n < 1:
        return

    for length in range(1, n + 1):
        paired_digits = length // 2
        weights = (2,) * paired_digits
        if length % 2:
            weights += (1,)

        minimum_sum = sum(weights)
        maximum_sum = 9 * minimum_sum
        if not minimum_sum <= n <= maximum_sum:
            continue

        for half in _halves_with_sum(weights, n):
            left = half[:paired_digits]
            if length % 2:
                palindrome = half + left[::-1]
            else:
                palindrome = half + half[::-1]
            yield int(palindrome)

# Claude Sonnet 5's Solution

In [8]:
def gen_digit_sequences(length, target):
    """
    Yield every list of `length` digits, each in 1..9, that sums to `target`.
    Uses feasibility pruning (min/max achievable sum) to avoid dead branches.
    """
    if length == 0:
        if target == 0:
            yield []
        return

    remaining_positions = length - 1
    min_possible = remaining_positions * 1
    max_possible = remaining_positions * 9

    for d in range(1, 10):
        rem = target - d
        if min_possible <= rem <= max_possible:
            for tail in gen_digit_sequences(remaining_positions, rem):
                yield [d] + tail


def claude_find_palindromes(n):
    """
    Return a sorted list of ints: all zero-free palindromes whose digits
    sum to n.
    """
    if n < 1:
        return []

    palindromes = []
    max_length = n  # each digit >= 1, so length can't exceed the target sum

    for L in range(1, max_length + 1):
        if L % 2 == 0:
            h = L // 2
            if n % 2 != 0:
                continue  # 2*sum(half) is always even
            target = n // 2
            for half in gen_digit_sequences(h, target):
                full_digits = half + half[::-1]
                palindromes.append(full_digits)
        else:
            h = (L + 1) // 2
            if h == 1:
                # single-digit palindrome: the digit itself must equal n
                if 1 <= n <= 9:
                    palindromes.append([n])
                continue
            # half = prefix (h-1 digits) + middle digit m
            # n = 2*sum(prefix) + m
            for m in range(1, 10):
                rem = n - m
                if rem < 0 or rem % 2 != 0:
                    continue
                prefix_target = rem // 2
                for prefix in gen_digit_sequences(h - 1, prefix_target):
                    half = prefix + [m]
                    full_digits = half + half[:-1][::-1]
                    palindromes.append(full_digits)

    numbers = sorted(int("".join(map(str, digits))) for digits in palindromes)
    return numbers

# Kimi K3's Solution

In [9]:
def kimi_palindromes_with_digit_sum(digit_sum, *, as_int=False):
    """
    Return every positive base-10 palindrome with no zero digits whose
    digit sum equals `digit_sum`.

    Returns a numerically sorted list of strings by default.
    Set as_int=True to return integers.
    """
    if not isinstance(digit_sum, int) or isinstance(digit_sum, bool):
        raise TypeError("digit_sum must be an integer")
    if digit_sum < 1:
        return []

    def compositions(total, parts):
        """Generate compositions of total into parts, each between 1 and 9."""
        if parts == 1:
            if 1 <= total <= 9:
                yield (total,)
            return

        lo = max(1, total - 9 * (parts - 1))
        hi = min(9, total - (parts - 1))

        for first in range(lo, hi + 1):
            for rest in compositions(total - first, parts - 1):
                yield (first,) + rest

    found = set()

    # Single-digit palindromes.
    if digit_sum <= 9:
        found.add(str(digit_sum))

    # Even-length palindromes: left + reverse(left).
    # These exist only when the digit sum is even.
    if digit_sum % 2 == 0:
        half_sum = digit_sum // 2

        for half_length in range(1, half_sum + 1):
            for left in compositions(half_sum, half_length):
                found.add("".join(map(str, left + left[::-1])))

    # Odd-length palindromes: left + center + reverse(left).
    # These must be checked for every digit sum because the center digit
    # can make the total digit sum either even or odd.
    for half_length in range(1, (digit_sum - 1) // 2 + 1):
        for center in range(1, 10):
            remaining = digit_sum - center

            if remaining < 0 or remaining % 2:
                continue

            left_sum = remaining // 2
            if not half_length <= left_sum <= 9 * half_length:
                continue

            for left in compositions(left_sum, half_length):
                digits = left + (center,) + left[::-1]
                found.add("".join(map(str, digits)))

    result = sorted(found, key=lambda x: (len(x), x))
    return [int(x) for x in result] if as_int else result

# Timing Results

Here we see the time each function takes to run the test suite.

In [10]:
%time test(palindromes_with_digit_sum)

CPU times: user 627 ms, sys: 68.1 ms, total: 695 ms
Wall time: 695 ms


In [11]:
%time test(gemini_generate_palindromes)

CPU times: user 7.31 s, sys: 96.8 ms, total: 7.41 s
Wall time: 7.42 s


In [12]:
%time test(chatGPT_palindromes_with_digit_sum)

CPU times: user 9.95 s, sys: 120 ms, total: 10.1 s
Wall time: 10.1 s


In [13]:
%time test(claude_find_palindromes)

CPU times: user 15.4 s, sys: 544 ms, total: 15.9 s
Wall time: 16.1 s


In [14]:
%time test(kimi_palindromes_with_digit_sum)

CPU times: user 14.6 s, sys: 261 ms, total: 14.8 s
Wall time: 14.9 s
